# 12. Laporan Metrik Uji

Notebook ini tidak memanggil API. Ia membaca hasil per item dari notebook 07-11 dan menyusun `results/laporan_uji.md`.

- E1 dan E2 adalah klasifikasi, jadi dilaporkan dengan keluarga F1: TP, FP, TN, FN, presisi, recall, F1.
- Cek Paham (ASR) dilaporkan dengan WER, metrik bakunya sendiri.

Jalankan setelah notebook 07-11 selesai.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import notebook_kit as kit
from src.config import RESULTS_DIR

BANDS = {"heuristik": (42, 70), "groq": (42, 70), "winston": (35, 70)}
LEVELS = list(range(1, 7))
REPORT = RESULTS_DIR / "laporan_uji.md"


def percent(df, columns):
    return df.assign(**{c: df[c].map(lambda v: "-" if pd.isna(v) else f"{v:.1%}") for c in columns})

## 1. E1: Deteksi AI

Kelas positif adalah AI. FP adalah tulisan manusia yang dituduh AI. Tiga cara membaca salah tuduh:

- FPR: dari semua tulisan manusia, berapa persen dituduh.
- FDR: dari semua teks yang ditandai AI, berapa persen sebenarnya tulisan manusia.
- FP / semua teks: salah tuduh dibagi seluruh teks yang dinilai.

Data gabungan = pilot notebook 07 (5 manusia, 5 AI) + lanjutan notebook 10 (40 manusia, 20 AI). Winston hanya menilai 18 teks manusia dan 5 esai AI karena kredit terbatas.

In [ ]:
pilot = pd.read_csv(RESULTS_DIR / "e1_esai" / "per_esai.csv").assign(data="pilot (07)")
lanjutan = pd.read_csv(RESULTS_DIR / "e1_esai_conversation" / "per_esai.csv").assign(data="lanjutan (10)")
e1 = pd.concat([pilot, lanjutan], ignore_index=True)
long = e1.melt(id_vars=["id", "label", "data"], value_vars=list(BANDS), var_name="detektor", value_name="skor").dropna()


def e1_table(part):
    rows = []
    for name, group in part.groupby("detektor"):
        ai = group.label == "ai"
        for rule, threshold in (("sedang", BANDS[name][0]), ("tinggi", BANDS[name][1])):
            flagged = group.skor >= threshold
            m = kit.binary_metrics(int((ai & flagged).sum()), int((~ai & flagged).sum()),
                                   int((~ai & ~flagged).sum()), int((ai & ~flagged).sum()))
            rows.append({"detektor": name, "aturan": f"{rule} ({threshold}+)", "manusia": int((~ai).sum()),
                         "AI": int(ai.sum()), **m})
    return pd.DataFrame(rows)


RATES = ["akurasi", "presisi", "recall (TPR)", "spesifisitas (TNR)", "FPR (salah tuduh / manusia)",
         "FDR (salah tuduh / yang ditandai)", "FP / semua teks", "F1"]
e1_tables = {"Gabungan": e1_table(long)}
for name, part in long.groupby("data", sort=False):
    e1_tables[name] = e1_table(part)
percent(e1_tables["Gabungan"], RATES)

In [ ]:
for name in ("pilot (07)", "lanjutan (10)"):
    print(name)
    display(percent(e1_tables[name], RATES))

## 2. E2: Level Bloom

Klasifikasi enam kelas (L1-L6). F1 makro = rata-rata F1 tiap level, sehingga level yang jarang ikut dihitung setara. F1 berbobot menimbang tiap level dengan jumlah datanya. Selisih maks. 1 dan kappa kuadrat ditambahkan karena level Bloom berurutan.

In [ ]:
bloom = pd.read_csv(RESULTS_DIR / "e2_bloom" / "per_jawaban.csv").query("ok == True")
soal = pd.read_csv(RESULTS_DIR / "e2_bloom_soal" / "per_soal.csv")

tests = {
    "Jawaban terkontrol (09), heuristik": (bloom.target_level, bloom.heuristik),
    "Jawaban terkontrol (09), Groq": (bloom.target_level, bloom.groq),
    "Soal Indonesia (11), Groq": (soal.query("dataset == 'indonesia'").level, soal.query("dataset == 'indonesia'").groq),
    "Soal Yahya (11), Groq": (soal.query("dataset == 'yahya'").level, soal.query("dataset == 'yahya'").groq),
    "Soal Indonesia (11), kamus kata kerja": (soal.level, soal["kamus Krathwohl"]),
}

summary_rows, per_level = [], {}
for name, (truth, pred) in tests.items():
    ok = pred.notna()
    truth, pred = truth[ok].astype(int).tolist(), pred[ok].astype(int).tolist()
    rows, summary = kit.multiclass_report(truth, pred, LEVELS)
    summary["selisih maks. 1"] = sum(abs(t - p) <= 1 for t, p in zip(truth, pred)) / len(truth)
    summary["kappa kuadrat"] = kit.quadratic_kappa(truth, pred)
    summary_rows.append({"uji": name, **summary})
    per_level[name] = [r["F1"] for r in rows]

e2 = pd.DataFrame(summary_rows)
e2_levels = pd.DataFrame(per_level, index=[f"L{i}" for i in LEVELS]).round(2)
display(percent(e2, ["akurasi", "presisi makro", "recall makro", "F1 makro", "F1 berbobot", "selisih maks. 1"]).round(3))
e2_levels

## 3. Cek Paham: ASR

WER (persen kata salah) dan CER (persen huruf salah) dihitung per korpus pada FLEURS id_id. Diambil dari notebook 08.

In [ ]:
asr = pd.read_csv(RESULTS_DIR / "asr_fleurs" / "ringkasan_model.csv", index_col=0)
asr_view = asr[["klip", "gagal", "WER", "WER bawah", "WER atas", "CER", "latensi median", "latensi p95", "RTF median"]].copy()
asr_view["WER (CI 95%)"] = [f"{w:.1%} ({lo:.1%}-{hi:.1%})" for w, lo, hi in zip(asr.WER, asr["WER bawah"], asr["WER atas"])]
asr_view["CER"] = asr.CER.map(lambda v: f"{v:.1%}")
asr_view = asr_view[["klip", "gagal", "WER (CI 95%)", "CER", "latensi median", "latensi p95", "RTF median"]].round(3)
asr_view

## 4. Laporan

In [ ]:
def table(df):
    return kit.md_table(df.round(3))


g = e1_tables["Gabungan"].set_index(["detektor", "aturan"])
lines = [
    f"- {d}: salah tuduh {g.loc[(d, r), 'FP']}/{g.loc[(d, r), 'manusia']} tulisan manusia "
    f"(FPR {g.loc[(d, r), 'FPR (salah tuduh / manusia)']:.1%}), F1 {g.loc[(d, r), 'F1']:.2f}."
    for d, r in g.index if r.startswith("sedang")
]

kit.write_summary(REPORT, "Laporan Uji ThinkPath: E1, E2, dan Cek Paham", {
    "Ringkas": "E1 pada ambang sedang, data gabungan:\n" + "\n".join(lines) + "\n\n"
               f"E2: F1 makro tertinggi {e2['F1 makro'].max():.2f} ({e2.loc[e2['F1 makro'].idxmax(), 'uji']}).\n\n"
               f"Cek Paham: WER terendah {asr.WER.min():.1%} ({asr.WER.idxmin()}).",
    "E1: deteksi AI (positif = AI)":
        "FPR = salah tuduh dari semua tulisan manusia. FDR = salah tuduh dari semua teks yang ditandai AI. "
        "Winston pada data gabungan hanya 18 teks manusia dan 5 esai AI.\n\n### Gabungan\n\n"
        + table(percent(e1_tables["Gabungan"], RATES))
        + "\n\n### Pilot (notebook 07)\n\n" + table(percent(e1_tables["pilot (07)"], RATES))
        + "\n\n### Lanjutan (notebook 10)\n\n" + table(percent(e1_tables["lanjutan (10)"], RATES)),
    "E2: level Bloom (6 kelas)":
        table(percent(e2, ["akurasi", "presisi makro", "recall makro", "F1 makro", "F1 berbobot", "selisih maks. 1"]))
        + "\n\n### F1 per level\n\n" + kit.md_table(e2_levels.reset_index(names="level")),
    "Cek Paham: ASR (WER)": table(asr_view.reset_index(names="model")),
    "Batasan":
        "- E1: tulisan manusia lanjutan berasal dari artikel akademisi yang disunting, bukan esai mahasiswa. "
        "Winston hanya menilai sebagian teks karena kredit terbatas.\n"
        "- E2: label notebook 09 adalah level yang diminta ke generator; label notebook 11 melekat pada soal, bukan jawaban. "
        "Kamus kata kerja hanya bisa dinilai pada soal yang memuat kata kerja.\n"
        "- Cek Paham: FLEURS berisi kalimat yang dibacakan dengan jelas, sedangkan jawaban Cek Paham spontan.",
})
print(REPORT.read_text())